# glbench × Colab — prefill bottleneck hunt

Two things in one notebook:

1. **Wire glbench to Colab.** Build the `glbench` binary (Mensura Veritatis) — it links `glcuda`, so it runs the CUDA engine through the real `GlEngine` contract and archives a `BenchmarkSession` (JSON) with analysis + bottleneck classification.
2. **Find the prefill culprit.** glbench reports the prefill *total* (it observes; it doesn't crack the kernel open). The per-op breakdown lives in glcuda's own profiler: `GLCUDA_PROFILE_PREFILL=1` prints `[prefill split]` (qkv / attn / ffn) plus two detail lines — `[attn detail]` (norm+rope / kv-write / attn core) and `[ffn detail]` (gate+up GEMM / down+o GEMM / elementwise). We run both, then parse + plot the split so the biggest sub-bucket is the next optimization target — measure, don't guess.

**Runtime:** GPU **T4** + Internet on. ~8 min incl. one 4.4 GB model download.

llama.cpp prefill baseline (earlier same-T4 runs, reference only — not re-run here): Q8_0 ~1400, Q4_K_M ~1200, Q4_0 ~1320 tok/s.

> **Why `[attn detail]` exists.** The first profile put attn at ~40% of prefill with no big GEMM inside it — the largest *uninstrumented* bucket, not yet a confirmed bottleneck. So attn was split (like FFN before it) into norm+rope / kv-write / attn-core. This run tells us whether the cost is `attn_decode_rows` (the decode-shaped core run over prefill rows), the rope/norm glue, or the kv-write traffic.

## 0 · GPU + Rust toolchain

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv
import os, subprocess
# Kaggle path adapter (harmless on Colab).
if os.path.exists('/kaggle') and not os.path.exists('/content'):
    subprocess.run(['ln','-sfn','/kaggle/working','/content'], check=True)
    print('linked /content -> /kaggle/working')
if not os.path.exists(os.path.expanduser('~/.cargo/bin/cargo')):
    !curl -sSf https://sh.rustup.rs | sh -s -- -y >/dev/null 2>&1
os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']
!cargo --version

## 1 · Clone + build the `glbench` binary (links glcuda, ~3-4 min)

`cargo build --release -p glbench` compiles glbench and its workspace deps (glcore, glproc, glcuda). glbench has **zero external deps** — only the Rust stdlib and workspace crates — so this is just the engine build plus a thin CLI.

In [ ]:
%cd /content
if not os.path.exists('/content/gwenland-ai'):
    !git clone -q https://github.com/JinXSuperSolo/gwenland-ai
%cd /content/gwenland-ai
!git pull --ff-only 2>&1 | tail -1
!cargo build --release -p glbench 2>&1 | tail -4
BIN = '/content/gwenland-ai/target/release/glbench'
assert os.path.exists(BIN), 'glbench binary not built'
!$BIN --help

## 2 · Model — Qwen2.5-7B Q8_0 (the FFN-profiling target)

In [ ]:
%cd /content/gwenland-ai
if not os.path.exists('q8.gguf'):
    os.system('wget -q -O q8.gguf https://huggingface.co/bartowski/Qwen2.5-7B-Instruct-GGUF/resolve/main/Qwen2.5-7B-Instruct-Q8_0.gguf')
!ls -lh q8.gguf

## 3 · glbench `run --kind prefill` — the archived, analyzed number

This is the clean measurement: warmup, N timed iterations, a `BenchmarkSession` archived to JSON with the analysis pass (health, ceiling efficiency, bottleneck classification). The prompt repeats to ~500 tokens so we exercise the prefill path like llama-bench's `pp512` and not launch overhead (glbench flags any prompt < 128 tokens as unrepresentative).

In [ ]:
%cd /content/gwenland-ai
BASE = 'Explain how quantum computing differs from classical computing, covering qubits, superposition, entanglement, and key algorithms. '
PROMPT = (BASE * 24).strip()   # ~500 tokens, like llama-bench pp512

r = subprocess.run(
    [BIN, 'run', '--engine', 'glcuda', '--model', 'q8.gguf',
     '--prompt', PROMPT, '--kind', 'prefill',
     '--warmup', '1', '--iters', '5',
     '--out', '/content/prefill_glcuda_q8.json'],
    capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print('--- stderr ---'); print(r.stderr[-2000:])

In [ ]:
# Pull the headline facts straight out of the archived session (hand-rolled JSON,
# but valid — Python's json parses it fine).
import json
with open('/content/prefill_glcuda_q8.json') as f:
    sess = json.load(f)
a = sess.get('analysis', {})
pf = a.get('prefill_tps', {})
print(f"prefill tok/s   : mean {pf.get('mean',0):.1f}  median {pf.get('median',0):.1f}  "
      f"min {pf.get('min',0):.1f}  max {pf.get('max',0):.1f}")
print(f"bottleneck      : {a.get('bottleneck','?')}")
eff = a.get('ceiling_efficiency')
print(f"ceiling eff     : {eff:.1%}" if eff is not None else 'ceiling eff     : (no ceiling)')
for n in a.get('notes', []):
    print('  •', n)

## 4 · ⭐ The culprit — glcuda's per-op prefill split

glbench sees the prefill total; it will not crack the kernel open (that's the one rule — it observes, it doesn't optimize). To localize *where inside prefill* the time goes, we run the same engine directly with `GLCUDA_PROFILE_PREFILL=1`, which syncs after each phase and prints three lines to stderr:

- `[prefill split]` — **qkv** (norm+quant+Q/K/V GEMMs) vs **attn** (bias/qk-norm/rope/kv-write/attention core) vs **ffn** (norm+quant+gate/up/down GEMMs+silu+residual)
- `[attn detail]` — **norm+rope** (bias+qk-norm+rope) vs **kv-write** vs **attn core** (`attn_decode_rows`)
- `[ffn detail]` — **gate+up GEMM** vs **down+o GEMM** vs **elementwise** glue

The profiler serializes the pipeline (diagnostic only, never in production), so its *absolute* tok/s is lower than section 3 — use section 3 for the headline number and this for the **proportions**.

We reuse the tiny `run_glcuda` example driver (same one the prefill-profile notebook uses); glbench itself doesn't expose the env-gated stderr split.

In [ ]:
%cd /content/gwenland-ai
# Build the profiler driver (env-gated stderr split isn't surfaced through glbench).
example = r'''
use std::time::Instant;
use glcore::engine_trait::{GlEngine, InferInput};
use glcuda::GlcudaEngine;
fn main() -> Result<(), Box<dyn std::error::Error>> {
    let path = std::env::args().nth(1).expect("usage: run_glcuda <model.gguf> [prompt]");
    let prompt = std::env::args().nth(2)
        .unwrap_or_else(|| "Explain what a GPU is in one sentence.".to_string());
    let mut engine = GlcudaEngine::new();
    engine.init()?;
    engine.load_model(&path)?;
    let ids = engine.encode_chat(&prompt)?;
    eprintln!("[prompt] {} tokens", ids.len());
    let input = InferInput { token_ids: ids, max_new_tokens: 8, temperature: 0.0,
        top_k: 40, top_p: 0.95, repeat_penalty: 1.1 };
    let out = engine.stream(input, &|_id,_p| {})?;
    println!("GLCUDA prefill {:.1} tok/s",
        out.prompt_tokens as f64 / (out.prefill_ms/1e3).max(1e-9));
    Ok(())
}
'''
os.makedirs('/content/gwenland-ai/glcuda/examples', exist_ok=True)
open('/content/gwenland-ai/glcuda/examples/run_glcuda.rs','w').write(example)
mani = '/content/gwenland-ai/glcuda/Cargo.toml'
txt = open(mani).read()
if 'name = "run_glcuda"' not in txt:
    open(mani,'a').write('\n[[example]]\nname = "run_glcuda"\npath = "examples/run_glcuda.rs"\n')
!cargo build --release -p glcuda --example run_glcuda 2>&1 | tail -2
DRIVER = 'target/release/examples/run_glcuda'

In [ ]:
%cd /content/gwenland-ai
env = {**os.environ, 'GLCUDA_PROFILE_PREFILL': '1'}
env.pop('GLCUDA_NO_MMA', None)
r = subprocess.run([DRIVER, 'q8.gguf', PROMPT], capture_output=True, text=True, env=env)

split_line = attn_line = ffn_line = None
for line in r.stderr.splitlines():
    if 'prefill split' in line:
        split_line = line; print(line)
    elif 'attn detail' in line:
        attn_line = line; print(line)
    elif 'ffn detail' in line:
        ffn_line = line; print(line)
    elif '[prompt]' in line:
        print(line)
for line in r.stdout.splitlines():
    if 'GLCUDA' in line:
        print(line)
if not split_line:
    print('--- no split captured; stderr tail ---'); print(r.stderr[-1500:])
elif not attn_line:
    print('--- no [attn detail] — is the engine build up to date? (git pull + rebuild) ---')

## 4c · Inside attn core — the pass split (Stage 2c evidence)

The profiler resolves `attn core` as one number; its three passes (QK scores / softmax / V-accum) are separated by `bar.sync` inside a single launch, so no host timing can split them. `gl_attn_rows_probe` (bench `[attn]` section) is an early-exit copy of the kernel that attributes the time by subtraction, at real 7B GQA shapes on a mid-prompt chunk.

The source-level prediction this measures: **the score pass dominates**, because its K reads are fully uncoalesced (~8× sector amplification, one thread per key row) while V-accum's reads are coalesced. The `full` vs `real kernel` numbers should match — that cross-checks the probe copy is faithful.

In [ ]:
%cd /content/gwenland-ai
!cargo build --release -p glcuda --example bench 2>&1 | tail -2
r = subprocess.run(['target/release/examples/bench'], capture_output=True, text=True)
for line in r.stdout.splitlines():
    if line.startswith('[attn]') or line.startswith('[bandwidth]') or line.startswith('device'):
        print(line)
if '[attn]' not in r.stdout:
    print('--- no [attn] section; stderr tail ---'); print(r.stderr[-1000:])

## 5 · Parse + visualize — name the culprit

Parse the three stderr lines into ms/percent and plot them. The biggest bar in each chart is the next thing to optimize; the attn split is the new signal this run adds.

In [ ]:
import re

def parse_buckets(line):
    # matches: '<label> <ms>ms (<pct>%)' repeated, e.g. 'qkv 12ms (18%)'.
    # label allows letters, '+', '-', and spaces (e.g. 'gate+up GEMM', 'norm+rope').
    out = []
    for label, ms, pct in re.findall(r'([A-Za-z+\- ]+?)\s+([\d.]+)ms\s+\(([\d.]+)%\)', line):
        out.append((label.strip(), float(ms), float(pct)))
    return out

split = parse_buckets(split_line) if split_line else []
attn  = parse_buckets(attn_line)  if attn_line  else []
ffn   = parse_buckets(ffn_line)   if ffn_line   else []

def show(title, buckets):
    if not buckets:
        print(f'\n{title}: (no data)'); return None
    print(f'\n{title}')
    for label, ms, pct in sorted(buckets, key=lambda b: -b[2]):
        bar = '█' * round(pct / 2)
        print(f'  {label:<16}{ms:>7.0f} ms  {pct:>4.0f}%  {bar}')
    return max(buckets, key=lambda b: b[2])

top_split = show('[prefill split]  coarse phases', split)
top_attn  = show('[attn detail]    attn sub-buckets', attn)
top_ffn   = show('[ffn detail]     FFN sub-buckets', ffn)
print()
if top_split:
    print(f'>> dominant phase : {top_split[0]}  ({top_split[2]:.0f}% of prefill)')
if top_attn:
    print(f'>> attn culprit   : {top_attn[0]}  ({top_attn[2]:.0f}% of prefill)')
if top_ffn:
    print(f'>> FFN culprit    : {top_ffn[0]}  ({top_ffn[2]:.0f}% of prefill)')

In [ ]:
# Three panels: coarse split, then the attn and ffn breakdowns. Skip any missing.
panels = [('prefill split', split), ('attn detail', attn), ('ffn detail', ffn)]
if any(b for _, b in panels):
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, (title, buckets) in zip(axes, panels):
        if not buckets:
            ax.set_axis_off(); ax.set_title(f'{title} (no data)'); continue
        buckets = sorted(buckets, key=lambda b: -b[2])
        labels = [b[0] for b in buckets]
        pcts   = [b[2] for b in buckets]
        bars = ax.barh(labels, pcts)
        bars[0].set_color('#d9534f')  # highlight the biggest bucket
        ax.invert_yaxis()
        ax.set_xlabel('% of prefill time')
        ax.set_title(title)
        for b, (_, ms, pct) in zip(bars, buckets):
            ax.text(pct + 1, b.get_y() + b.get_height()/2,
                    f'{pct:.0f}%  ({ms:.0f}ms)', va='center', fontsize=9)
        ax.set_xlim(0, max(pcts) * 1.35)
    plt.tight_layout()
    plt.savefig('/content/prefill_bottleneck.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('saved /content/prefill_bottleneck.png')
else:
    print('no profiler data to plot')

## 6 · (Optional) export the glbench archive to Markdown / CSV

The archived `BenchmarkSession` from section 3 is the source of truth. Convert it for a report or a spreadsheet.

In [ ]:
!$BIN export /content/prefill_glcuda_q8.json --format md  --out /content/prefill_glcuda_q8.md
!$BIN export /content/prefill_glcuda_q8.json --format csv --out /content/prefill_glcuda_q8.csv
print(open('/content/prefill_glcuda_q8.md').read())